## Работа с пропусками

### Что такое пропуски (missing values)

* Значения, которых нет в данных: `NaN/None`, пустые строки, иногда специальные значения типа `-999`, `0`.
* Почему возникают: ошибки сбора, несовпадение схем, фильтры анкет, отказ пациента/датчика, разные источники.

### Механизмы пропусков

* **MCAR** (полностью случайные): вероятность пропуска не зависит ни от чего. Простые методы работают.
* **MAR** (случайные при условии наблюдаемого): зависят от других известных признаков. Требуется условное/модельное заполнение.
* **MNAR** (не случайные): зависят от самого пропущенного значения (напр., не указывают вес, если он высокий). Самый сложный случай; простое заполнение может искажать выводы.

---

### Как находить пропуски

* Подсчёт: по столбцам/строкам и долям: `isnull().sum()`, доля = `mean()`.
* Проверка «скрытых» пропусков: пустые строки, специальные значения → сначала привести к `NaN`.
* Визуализация:

  * barchart долей пропусков.
  * Heatmap пропусков (паттерны по строкам).
  * Корреляции пропусков между признаками (помогает понять MAR).
* Анализ паттернов: совместные пропуски групп столбцов; сравнение распределений «есть значение» vs «нет значения» (первые намёки на MAR/MNAR).

---

### Как обрабатывать пропуски

#### 1) Оставить как есть

* Полезно, если модель умеет с `NaN` (не все) или если планируется специальная обработка.
* Часто добавляют **индикатор пропуска** (`feature_isna = 1/0`), особенно в медицине — сам факт отсутствия может быть информативен.

#### 2) Удаление

* **Строк**: когда доля пропусков мала и выборка большая.
* **Столбцов**: если признак почти весь пустой и не критичен.
* Риск: потеря информации, смещение выборки.

#### 3) Простое заполнение

* Числовые: константа, **медиана** (устойчива к выбросам) или среднее.
* Категориальные: **мода** или отдельная категория `"Unknown"`.
* По группам: заполнять статистикой внутри клинических групп (пол, возрастная группа и т.п.) или кластеров.

#### 4) Модельная импутация

* **KNNImputer**: ищет похожих пациентов.
* **MICE/IterativeImputer**: поочерёдно предсказывает каждый признак из остальных (лучше при MAR).
* **Временные ряды**: `ffill/bfill`, линейная/сплайн-интерполяция, модели (ARIMA, Kalman).

#### 5) Специальные случаи

* Когда «0» означает «не измеряли» → сначала заменить на `NaN`, потом заполнить.
* При **MNAR**: рассмотреть чувствительный анализ, множественное заполнение с разными сценариями, сохранить индикаторы пропусков, обсудить с доменными экспертами.


## Работа с выбросами

### Что такое выбросы (outliers)

* **Определение**: значения, сильно отличающиеся от большинства наблюдений.
* **Причины появления**:

  * Ошибки измерений (датчик, перенос данных).
  * Ошибки ввода, опечатка (например, рост = 20 см).
  * Редкие, но реальные наблюдения (человек весом 200 кг).
  * Естественная вариативность в «тяжёлых хвостах» распределения.

---

## Как находить выбросы

### 1. Визуально

* **Boxplot (ящик с усами)**: точки за пределами усов = кандидаты в выбросы.
* **Scatterplot**: помогает увидеть аномалии в двухмерных признаках.
* **Histogram/Distplot**: длинные хвосты, пики.

### 2. Статистические методы

* **IQR (межквартильный размах)**: выбросы = значения < Q1 − 1.5*IQR или > Q3 + 1.5*IQR.
* **z-score**: выбросы = |z| > 3 (или 2.5 в маленьких выборках).
* **Правило 3 сигм**: аналогично z-score, но проще (среднее ± 3σ).

---

## Как работать с выбросами

### 1. Удаление

* Подходит для явных ошибок (рост = 5 см, давление = 500).
* Риск: потеря редких, но важных наблюдений.

### 2. Замена

* Заменить на ближайшую границу нормального диапазона.
* Заменить константой, медианой или предсказанием модели.

### 3. Сохранение выбросов

* Если выбросы — **реальные данные**, их нельзя удалять (например, редкие болезни). «Аномалия» может быть важнее, чем среднее значение
* Часто добавляют индикатор: `is_outlier = 1/0`. Не «чистить» автоматически: фиксировать, почему и чем заменили.
* В моделях: можно использовать **устойчивые алгоритмы** (RobustScaler, деревья решений, бустинг).

### Выбор подхода зависит от:
- Природы данных (числовые, категориальные, временные ряды и т.д.).
- Целей анализа (обнаружение закономерностей, построение моделей).
- Доли пропусков или выбросов.
- Контекста задачи (например, клинические данные могут требовать более строгой обработки).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest

In [2]:
data = pd.read_csv('медицинские_данные.csv')

In [3]:
data.head(5)

,ID_пациента,ФИО,Возраст,Пол,Рост_см,Вес_кг,Систолическое_АД,Диастолическое_АД,Уровень_холестерина,Диагноз
0,1,Алексей Горшков,NaN,М,169.0,98.0,NaN,103.0,NaN,Диабет
1,2,Анна Майорова,29.0,Ж,168.0,NaN,NaN,91.0,70.0,Диабет
2,3,Елена Кузнецова,53.0,Ж,NaN,93.0,161.0,NaN,70.0,Ожирение
3,4,Олег Иванов,NaN,М,NaN,119.0,115.0,96.0,80.0,Диабет
4,5,Петр Иванов,NaN,М,173.0,NaN,108.0,NaN,90.0,Сердечная недостаточность


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_пациента          1000 non-null   int64  
 1   ФИО                  1000 non-null   object 
 2   Возраст              471 non-null    float64
 3   Пол                  1000 non-null   object 
 4   Рост_см              511 non-null    float64
 5   Вес_кг               516 non-null    float64
 6   Систолическое_АД     496 non-null    float64
 7   Диастолическое_АД    521 non-null    float64
 8   Уровень_холестерина  494 non-null    float64
 9   Диагноз              1000 non-null   object 
dtypes: float64(6), int64(1), object(3)
memory usage: 78.3+ KB


In [5]:
# === Поиск пропусков ===

# 1. Подсчет количества пропусков в каждом столбце
data.isnull().sum()

ID_пациента              0
ФИО                      0
Возраст                529
Пол                      0
Рост_см                489
Вес_кг                 484
Систолическое_АД       504
Диастолическое_АД      479
Уровень_холестерина    506
Диагноз                  0
dtype: int64

In [ ]:
# 1.1. Выявление строк с хотя бы одним пропуском
mis_df = data[data.isnull().any(axis=1)]
mis_df

In [ ]:
# доля строк с пропусками
mis_df.shape[0]/data.shape[0] * 100

In [ ]:
# 2. Визуализация пропусков
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title("Тепловая карта пропусков")
plt.show()

In [ ]:
import missingno as msno
msno.matrix(data)

In [ ]:
msno.bar(data)

In [ ]:
msno.heatmap(data)

In [ ]:
msno.dendrogram(data)

In [ ]:
# === Поиск выбросов ===
# Визуализация

In [ ]:
numeric_cols = data.select_dtypes(include="number").columns.drop("ID_пациента")

print(numeric_cols)

In [ ]:
# Гистограммы распределения данных
for column in numeric_cols:
    data[column].hist(bins=20)
    plt.title(f"Гистограмма: {column}")
    plt.xlabel(column)
    plt.ylabel("Частота")
    plt.show()

In [ ]:
for column in numeric_cols:
    sns.scatterplot(data[column])
    plt.title(f"Scatterplot: {column}")
    plt.show()

In [ ]:
sns.pairplot(data[numeric_cols], diag_kind="hist")
plt.show()

In [ ]:
# Boxplot
for c in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(y=data[c])
    plt.title(f"Boxplot: {c}")
    plt.show()

In [ ]:
# 1. Метод межквартильного размаха (IQR)
def iqr_outlier_mask(column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_iqr = (data[column] < lower_bound) | (data[column] > upper_bound)
    print(f"Количество выбросов в столбце {column} (IQR): {sum(outliers_iqr)}")
    return outliers_iqr

for column in numeric_cols:
    iqr_outlier_mask(column)

In [ ]:
# Визуализация: подсветим точки, которые IQR считает выбросами
col_w = "Вес_кг"
col_h = "Рост_см"
mask_iqr_h = iqr_outlier_mask(col_h)
mask_iqr_w = iqr_outlier_mask(col_w)
mask_iqr_any = (mask_iqr_h | mask_iqr_w)
sns.scatterplot(x=data[col_h], y=data[col_w], hue=mask_iqr_any.map({True:"IQR outlier", False:"normal"}))
plt.title("Рост vs Вес с подсветкой IQR-выбросов")
plt.show()

In [ ]:
data_out = data.copy()
for column in numeric_cols:
    data_out[column] = data_out[column].fillna(data_out[column].median())
data_out.head()

In [ ]:
# 2. Метод z-оценки
for column in numeric_cols:
    data[f"z_score_{column}"] = zscore(data_out[column])
    выбросы_z = data[abs(data[f"z_score_{column}"]) > 3]
    print(f"Количество выбросов в столбце {column} (z-оценка): {len(выбросы_z)}")

In [ ]:
# 3. Метод 3 сигм
for column in ["Возраст", "Рост_см", "Вес_кг", "Систолическое_АД", "Диастолическое_АД", "Уровень_холестерина"]:
    mean = data_out[column].mean()
    std = data_out[column].std()
    outliers_3sigma = data_out[(data_out[column] < mean - 3 * std) | (data_out[column] > mean + 3 * std)]
    print(f"Количество выбросов в столбце {column} (3 сигмы): {len(outliers_3sigma)}")

In [ ]:
# 4. Использование Isolation Forest для многомерных выбросов
clf = IsolationForest(contamination=0.025, random_state=42)
data_out["is_outlier"] = clf.fit_predict(data_out[["Рост_см", "Вес_кг"]])
outliers_iforest = data_out[data_out["is_outlier"] == -1]
print("Количество выбросов (Isolation Forest):", len(outliers_iforest))

In [ ]:
labels = clf.fit_predict(data_out[["Рост_см", "Вес_кг"]])   # 1 = норм, -1 = выброс
labels

In [ ]:
scores = -clf.score_samples(data_out[["Рост_см", "Вес_кг"]])  # чем выше, тем «аномальнее»
scores

In [ ]:
data_if = data_out.copy()
data_if["IF_label"] = labels
data_if["IF_score"] = scores

print("Isolation Forest выбросы:", (labels == -1).sum())

In [ ]:
# Визуализация
sns.scatterplot(x=data_if[col_h], y=data_if[col_w],
                hue=data_if["IF_label"].map({1:"normal", -1:"IF outlier"}))
plt.title("Рост vs Вес — Isolation Forest")
plt.show()

In [ ]:
# ТОП-аномалии по score (полезно для ручной проверки)
display(data_if.sort_values("IF_score", ascending=False).head(30)[[col_h, col_w, "IF_score"]])

In [ ]:
# 4. Использование Isolation Forest для многомерных выбросов
clf = IsolationForest(contamination=0.02)
data_out["w_is_outlier"] = clf.fit_predict(data_out[["Рост_см"]])
outliers_iforest = data_out[data_out["w_is_outlier"] == -1]
print("Количество выбросов по росту(Isolation Forest):", len(outliers_iforest))

In [ ]:
# 4. Использование Isolation Forest для многомерных выбросов
clf = IsolationForest(contamination=0.02)
data_out["h_is_outlier"] = clf.fit_predict(data_out[["Вес_кг"]])
outliers_iforest = data_out[data_out["h_is_outlier"] == -1]
print("Количество выбросов по весу (Isolation Forest):", len(outliers_iforest))

In [ ]:
# === Заполнение пропусков ===

# 1. Заполнение фиксированным значением (например, 0)
data_filled_fixed = data.fillna(0)
data_filled_fixed.head()

In [ ]:
# 2. Заполнение медианными значениями
data_new = data.copy()
for column in numeric_cols:
    data_new.fillna({column: data_new[column].median()}, inplace=True)
data_new.head()

In [ ]:
data_new2 = data.copy()
df_median = data_new2.groupby(['Пол', 'Диагноз'])[numeric_cols].median()
df_median

In [ ]:
data_new3 = data_new2.set_index(['Пол', 'Диагноз']).fillna(df_median).reset_index()
data_new3 = data_new3[data.columns]
data_new3

In [ ]:
# 3. Заполнение с использованием forward fill (предыдущее значение)
data_filled_ffill = data.ffill()
data_filled_ffill

In [ ]:
# 4. Заполнение с использованием backward fill (следующее значение)
data_filled_bfill = data.bfill()
data_filled_bfill

In [ ]:
data.iloc[[121, 142, 374, 393, 504, 541, 559, 604, 832, 971, 989, 170, 276, 395, 618, 897]]

In [ ]:
# === Заполнение выбросов ===

# 1. Заполнение выбросов медианой (метод IQR)
def replace_outliers_with_median(data, column):
    print(column)
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    median_value = data[column].median()
    print(median_value)
    data[column] = np.where((data[column] < lower_bound) | (data[column] > upper_bound), median_value, data[column])

data_copy = data.copy()
for column in ["Рост_см", "Вес_кг"]:
    replace_outliers_with_median(data_copy, column)
data_copy.head()

In [ ]:
data_copy.iloc[[121, 142, 374, 393, 504, 541, 559, 604, 832, 971, 989, 170, 276, 395, 618, 897]]

In [ ]:
# 2. Заполнение выбросов фиксированным значением
def replace_outliers_with_fixed(data, column, fixed_value):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    data[column] = np.where((data[column] < lower_bound) | (data[column] > upper_bound), fixed_value, data[column])

data_copy = data.copy()

replace_outliers_with_fixed(data_copy, "Рост_см", 160)  
replace_outliers_with_fixed(data_copy, "Вес_кг", 100)  # Пример: фиксированное значение 100

data_copy.head()

In [ ]:
data_copy.iloc[[121, 142, 374, 393, 504, 541, 559, 604, 832, 971, 989, 170, 276, 395, 618, 897]]

In [ ]:
# 3. Удаление выбросов
data_copy = data.copy()
for column in ["Рост_см", "Вес_кг"]:
    Q1 = data_copy[column].quantile(0.25)
    Q3 = data_copy[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    data_copy = data_copy[data_copy[column].isna() | ((data_copy[column] >= lower_bound) & (data_copy[column] <= upper_bound))]
data_copy.head()

In [ ]:
data_copy.shape

In [ ]:
# 4. Замена выбросов на среднее значение (метод z-оценки)
def replace_outliers_with_mean(data, column):
    mean_value = data[column].mean()
    z_scores = zscore(data[column].fillna(mean_value))
    data[column] = np.where(abs(z_scores) > 3, mean_value, data[column])

data_copy = data.copy()
for column in ["Рост_см", "Вес_кг"]:
    replace_outliers_with_mean(data_copy, column)
data_copy.head()